In [1]:
import torch
from models.magnifier import MagnifierModel as magnifier

In [4]:
# ------------------------------------------------------------------
# Usage Examples and Testing
# ------------------------------------------------------------------

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing Enhanced 2D FNO Magnifier on device: {device}\n")
    print("=" * 70)

    # ------------------------------------------------------------------
    # Example 1: Enhanced Configuration with Progressive Expansion
    # ------------------------------------------------------------------
    print("\n[Example 1] Enhanced Model with Progressive Channel Expansion")
    print("-" * 70)

    # Typical real-world shapes
    nb = 32               # batch size
    N = 5                # coarse patch size
    f = 4                # upscaling factor
    P_fine = N * f       # 40
    nt = 88              # time steps
    C_in = 2             # channels: coarse u interp + fine bed

    # Create dummy patch input
    patch_input = torch.randn(nb, C_in, P_fine, P_fine, nt, device=device)



    model_enhanced = magnifier(
        in_channels=C_in,
        base_channels=32,
        num_fno_blocks=4,
        fno_modes_x=6,
        fno_modes_y=6,
        num_refinement_blocks=4,
        num_residual_per_block=3,
        channel_multipliers=[1.0, 1.5, 2, 2],  # 48→64→80→96→96
        dropout=0.1,
        use_attention=False,
        use_pyramid_pooling=True,
        use_gradient_checkpointing=False
    ).to(device)
    
    output = model_enhanced(patch_input)
    
    print(f"Input shape:  {patch_input.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Channel progression: {model_enhanced.channel_counts}")
    
    total_params = sum(p.numel() for p in model_enhanced.parameters())
    print(f"Total parameters: {total_params:,}")
    torch.cuda.empty_cache()


Testing Enhanced 2D FNO Magnifier on device: cuda


[Example 1] Enhanced Model with Progressive Channel Expansion
----------------------------------------------------------------------
Input shape:  torch.Size([32, 2, 20, 20, 88])
Output shape: torch.Size([32, 1, 20, 20, 88])
Channel progression: [32, 32, 48, 64, 64]
Total parameters: 2,441,087
